# Colab timing test

Runs the LoRA smoke test on a Colab GPU to measure sec/step.
That number tells us if one training seed fits in about 3 hours.

Before running:
1. Runtime > Change runtime type > **T4 GPU**.
2. Add a GitHub token as a Colab secret named `GH_TOKEN` (key icon on the left). The repo is private.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import userdata
import os
token = userdata.get("GH_TOKEN")
if not os.path.isdir("ask-toronto"):
    !git clone -q https://{token}@github.com/Tantrayoga/ask-toronto.git
%cd ask-toronto
!git log -1 --format='%h %s'

Use Colab's own CUDA torch. The PyPI torch pin in `pyproject.toml` is the Mac build.
Pin the HF stack to the same versions as the Mac so results compare.

In [ ]:
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 accelerate==1.15.0 datasets sentencepiece==0.2.3

In [ ]:
import torch, transformers, peft, trl, accelerate, sys
print(sys.version.split()[0], "torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__, "peft", peft.__version__, "trl", trl.__version__, "accelerate", accelerate.__version__)

fp16 because the T4 has no bf16. The script pulls the model from the Hub on first run (about 3 GB).

In [ ]:
!PYTHONPATH=src python scripts/smoke_lora.py --dtype fp16

Copy back the last lines: loss by step, sec/step, peak memory and the three time estimates.

If fp16 shows NaN or inf loss, rerun with `--dtype fp32`.